# Data Leakage and Price Stickiness Analysis

This notebook investigates two potential concerns with the model performance:
1. **Data Leakage**: Are we accidentally using future information to predict the target?
2. **Price Stickiness**: Are freight prices "sticky" (unchanging for weeks), making prediction artificially easy?

**Context**: The best model (Linear Regression with engineered features) achieves RMSE of $178.89, which seems very good.
We need to determine if this is due to:
- Data leakage in feature engineering
- Price stickiness (prices staying the same for weeks)
- Legitimate model performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported")

## Part 1: Load Data and Basic Statistics

In [ ]:
# Load model data
df = pd.read_csv('data/processed/model_data.csv', parse_dates=['Date'], index_col='Date')
print(f"Loaded {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

# Get price column
price_col = 'Europe_Base_Price'
if price_col not in df.columns:
    raise ValueError(f"{price_col} not found in dataset")

# Basic statistics
print(f"\n📊 Price Statistics:")
print(f"  Mean: ${df[price_col].mean():.2f}")
print(f"  Std: ${df[price_col].std():.2f}")
print(f"  Min: ${df[price_col].min():.2f}")
print(f"  Max: ${df[price_col].max():.2f}")
print(f"  Range: ${df[price_col].max() - df[price_col].min():.2f}")

## Part 2: Price Stickiness Analysis

Investigate if freight prices stay constant for multiple weeks at a time.

In [ ]:
print("="*80)
print("PRICE STICKINESS ANALYSIS")
print("="*80)

# Calculate week-over-week price changes
df['price_change'] = df[price_col].diff()
df['price_change_pct'] = df[price_col].pct_change() * 100
df['price_abs_change'] = df['price_change'].abs()

# Count weeks with zero change
zero_change = (df['price_change'] == 0).sum()
total_weeks = len(df) - 1  # Exclude first row (no prior week)

# Count weeks with very small changes (<$10)
small_change = (df['price_abs_change'] < 10).sum()

# Count weeks with very small % changes (<1%)
small_pct_change = (df['price_change_pct'].abs() < 1.0).sum()

print(f"\n📍 Stickiness Metrics:")
print(f"  Weeks with ZERO change: {zero_change}/{total_weeks} ({zero_change/total_weeks*100:.1f}%)")
print(f"  Weeks with <$10 change: {small_change}/{total_weeks} ({small_change/total_weeks*100:.1f}%)")
print(f"  Weeks with <1% change: {small_pct_change}/{total_weeks} ({small_pct_change/total_weeks*100:.1f}%)")

# Detailed change statistics
print(f"\n📈 Price Change Statistics:")
print(f"  Mean absolute change: ${df['price_abs_change'].mean():.2f}")
print(f"  Median absolute change: ${df['price_abs_change'].median():.2f}")
print(f"  Std of changes: ${df['price_change'].std():.2f}")
print(f"  Max increase: ${df['price_change'].max():.2f}")
print(f"  Max decrease: ${df['price_change'].min():.2f}")

# Distribution of changes
print(f"\n📊 Distribution of Price Changes:")
print(df['price_abs_change'].describe())

In [ ]:
# Visualize price changes over time
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Plot 1: Actual prices
ax1 = axes[0]
ax1.plot(df.index, df[price_col], linewidth=1.5, color='blue')
ax1.set_title('Europe Base Port Prices Over Time', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD)', fontsize=12)
ax1.grid(alpha=0.3)

# Plot 2: Week-over-week absolute changes
ax2 = axes[1]
ax2.bar(df.index, df['price_abs_change'], color='orange', alpha=0.7)
ax2.axhline(y=10, color='red', linestyle='--', label='$10 threshold', linewidth=2)
ax2.set_title('Week-over-Week Absolute Price Changes', fontsize=14, fontweight='bold')
ax2.set_ylabel('Absolute Change (USD)', fontsize=12)
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Percentage changes
ax3 = axes[2]
ax3.bar(df.index, df['price_change_pct'], color='green', alpha=0.7)
ax3.axhline(y=1, color='red', linestyle='--', label='1% threshold', linewidth=2)
ax3.axhline(y=-1, color='red', linestyle='--', linewidth=2)
ax3.set_title('Week-over-Week Percentage Price Changes', fontsize=14, fontweight='bold')
ax3.set_ylabel('Change (%)', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualizations complete")

## Part 3: Autocorrelation Analysis

Measure how predictable prices are based on their own history.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf

print("="*80)
print("AUTOCORRELATION ANALYSIS")
print("="*80)

# Calculate autocorrelation for lags 1-8 weeks
autocorr_values = acf(df[price_col].dropna(), nlags=8)

print(f"\n📊 Autocorrelation by Lag:")
for lag in range(1, 9):
    print(f"  Lag {lag}w: {autocorr_values[lag]:.4f}")

# Plot ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(df[price_col].dropna(), lags=20, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lag (weeks)', fontsize=12)

plot_pacf(df[price_col].dropna(), lags=20, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lag (weeks)', fontsize=12)

plt.tight_layout()
plt.show()

# Interpretation
if autocorr_values[1] > 0.95:
    print(f"\n⚠️  WARNING: Very high autocorrelation (lag-1: {autocorr_values[1]:.4f})")
    print("   This suggests prices are HIGHLY predictable from the previous week.")
    print("   This could explain the low RMSE - a naive model would perform well.")
elif autocorr_values[1] > 0.85:
    print(f"\n✓ Moderate-high autocorrelation (lag-1: {autocorr_values[1]:.4f})")
    print("  Prices show persistence but still have meaningful variation.")
else:
    print(f"\n✓ Low autocorrelation (lag-1: {autocorr_values[1]:.4f})")
    print("  Prices change significantly week-to-week.")

## Part 4: Naive Baseline Performance

Test how well a naive "last week's price" model would perform.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("="*80)
print("NAIVE BASELINE PERFORMANCE")
print("="*80)

# Create naive predictions (use current week to predict next week)
df['naive_prediction'] = df[price_col].shift(1)
df['actual_next_week'] = df[price_col].shift(-1)

# Remove NaN rows
df_valid = df.dropna(subset=['naive_prediction', 'actual_next_week'])

# Calculate metrics
rmse_naive = np.sqrt(mean_squared_error(df_valid['actual_next_week'], df_valid['naive_prediction']))
mae_naive = mean_absolute_error(df_valid['actual_next_week'], df_valid['naive_prediction'])
mape_naive = np.mean(np.abs((df_valid['actual_next_week'] - df_valid['naive_prediction']) / df_valid['actual_next_week'])) * 100

print(f"\n🎯 Naive Model Performance (entire dataset):")
print(f"  RMSE: ${rmse_naive:.2f}")
print(f"  MAE: ${mae_naive:.2f}")
print(f"  MAPE: {mape_naive:.2f}%")

# Compare to best model
best_model_rmse = 178.89  # From notebook 04
improvement = ((rmse_naive - best_model_rmse) / rmse_naive) * 100

print(f"\n📊 Comparison to Best Model:")
print(f"  Best Model RMSE: ${best_model_rmse:.2f}")
print(f"  Naive Model RMSE: ${rmse_naive:.2f}")
print(f"  Improvement: {improvement:.1f}%")

if improvement < 15:
    print(f"\n⚠️  WARNING: Model only {improvement:.1f}% better than naive baseline!")
    print("   This suggests the features may not be adding much value.")
elif improvement < 30:
    print(f"\n✓ Model shows moderate improvement ({improvement:.1f}%) over naive baseline.")
else:
    print(f"\n✅ Model shows strong improvement ({improvement:.1f}%) over naive baseline!")
    print("   The features are adding significant predictive value.")

## Part 5: Data Leakage Detection

Check if any features have suspiciously high correlations with the target.

In [ ]:
print("="*80)
print("DATA LEAKAGE DETECTION")
print("="*80)

# Check for features that are too highly correlated with target
if 'price_1w_ahead' in df.columns:
    # Get numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # Calculate correlations with target
    correlations = df[numeric_cols].corrwith(df['price_1w_ahead']).abs().sort_values(ascending=False)
    
    # Filter out the target itself
    correlations = correlations[correlations.index != 'price_1w_ahead']
    
    # Check for suspiciously high correlations (>0.99)
    suspicious = correlations[correlations > 0.99]
    
    print(f"\n🔍 Top 20 Features by Correlation with Target:")
    print(correlations.head(20))
    
    if len(suspicious) > 0:
        print(f"\n⚠️  WARNING: {len(suspicious)} features have >0.99 correlation with target!")
        print("   These features may be leaking future information:")
        for feat in suspicious.index:
            print(f"     - {feat}: {suspicious[feat]:.6f}")
        print("\n   ⚠️  POTENTIAL DATA LEAKAGE DETECTED!")
    else:
        print(f"\n✓ No features with >0.99 correlation found.")
        
        # Check for very high correlations (>0.95)
        very_high = correlations[(correlations > 0.95) & (correlations <= 0.99)]
        if len(very_high) > 0:
            print(f"\n⚠️  Note: {len(very_high)} features have 0.95-0.99 correlation:")
            for feat in very_high.index[:5]:  # Show top 5
                print(f"     - {feat}: {very_high[feat]:.4f}")
            print("   These should be investigated but may be legitimate.")
        else:
            print("   No concerning correlations detected.")
else:
    print("⚠️  Target variable 'price_1w_ahead' not found in dataset!")

## Part 6: Feature Engineering Audit

Check specific features that were created in notebook 04 for potential leakage.

In [ ]:
print("="*80)
print("FEATURE ENGINEERING AUDIT")
print("="*80)

# Check for features that might use current or future data
suspicious_patterns = [
    ('Europe_Base_Price', 'Current price (should be excluded)'),
    ('SCFI_Index', 'Shanghai index (needs lag check)'),
    ('_lag_0w', 'Zero-lag feature (current week)'),
    ('price_rolling_', 'Rolling features (check if properly lagged)'),
    ('price_ema_', 'EMA features (check if properly lagged)'),
]

print(f"\n🔍 Checking for potentially problematic features:\n")

all_cols = df.columns.tolist()
issues_found = False

for pattern, description in suspicious_patterns:
    matching = [col for col in all_cols if pattern in col]
    if matching:
        print(f"⚠️  Found '{pattern}' features: {len(matching)}")
        print(f"   Description: {description}")
        print(f"   Examples: {matching[:3]}")
        issues_found = True
        print()

if not issues_found:
    print("✓ No obvious problematic feature patterns found.")

# Check if features are from notebook 04
print(f"\n📋 Feature Categories in Dataset:")
lag_features = [col for col in all_cols if '_lag_' in col]
rolling_features = [col for col in all_cols if 'rolling' in col.lower()]
ema_features = [col for col in all_cols if 'ema' in col.lower()]
momentum_features = [col for col in all_cols if 'momentum' in col.lower() or 'roc' in col.lower()]

print(f"  Lagged features: {len(lag_features)}")
print(f"  Rolling features: {len(rolling_features)}")
print(f"  EMA features: {len(ema_features)}")
print(f"  Momentum features: {len(momentum_features)}")

## Part 7: Conclusion and Diagnosis

In [ ]:
print("="*80)
print("DIAGNOSIS: ROOT CAUSE OF LOW RMSE")
print("="*80)

# Gather all metrics
print(f"\n📊 Summary of Findings:\n")

print(f"1. PRICE STICKINESS:")
print(f"   - Weeks with zero change: {zero_change/total_weeks*100:.1f}%")
print(f"   - Weeks with <$10 change: {small_change/total_weeks*100:.1f}%")
print(f"   - Lag-1 autocorrelation: {autocorr_values[1]:.4f}")

print(f"\n2. NAIVE BASELINE:")
print(f"   - Naive RMSE: ${rmse_naive:.2f}")
print(f"   - Best Model RMSE: ${best_model_rmse:.2f}")
print(f"   - Improvement: {improvement:.1f}%")

print(f"\n3. DATA LEAKAGE:")
if len(suspicious) > 0:
    print(f"   - Features with >0.99 correlation: {len(suspicious)}")
else:
    print(f"   - No obvious leakage detected")

# Make diagnosis
print(f"\n" + "="*80)
print(f"FINAL DIAGNOSIS")
print("="*80 + "\n")

# Determine primary cause
if len(suspicious) > 0:
    print("🔴 PRIMARY CAUSE: DATA LEAKAGE")
    print(f"   Features with >0.99 correlation suggest future information is leaking.")
    print(f"   RECOMMENDATION: Fix feature engineering in notebook 04.")
elif autocorr_values[1] > 0.95 and small_change/total_weeks > 0.3:
    print("🟡 PRIMARY CAUSE: PRICE STICKINESS")
    print(f"   Very high autocorrelation ({autocorr_values[1]:.4f}) and {small_change/total_weeks*100:.1f}% of weeks")
    print(f"   have <$10 changes. Prices are inherently sticky.")
    print(f"   This is a characteristic of container freight contracts, not a problem.")
    print(f"   RECOMMENDATION: Test with longer forecast horizons (e.g., 1 month ahead).")
elif improvement > 25:
    print("🟢 PRIMARY CAUSE: LEGITIMATE MODEL PERFORMANCE")
    print(f"   Model shows {improvement:.1f}% improvement over naive baseline.")
    print(f"   No significant leakage detected. Performance appears legitimate.")
    print(f"   RECOMMENDATION: Validate with out-of-sample testing and longer horizons.")
else:
    print("🟡 MIXED FACTORS")
    print(f"   Both stickiness and model features contribute to performance.")
    print(f"   RECOMMENDATION: Test with 1-month ahead predictions for better validation.")

print(f"\n" + "="*80)

## Part 8: Recommendations

Based on the analysis, here are the next steps to validate model performance.

In [ ]:
print("="*80)
print("RECOMMENDATIONS")
print("="*80)

print(f"\n✅ NEXT STEPS:\n")

print("1. CREATE 1-MONTH AHEAD PREDICTIONS")
print("   - Train models to predict 4 weeks ahead instead of 1 week")
print("   - Compare performance: Linear Regression, Decision Tree, KNN")
print("   - Expected: RMSE should increase significantly if stickiness is the issue")

print("\n2. VALIDATE FEATURE ENGINEERING")
print("   - Audit all rolling/EMA features to ensure proper lagging")
print("   - Remove any features with >0.95 correlation to target")
print("   - Verify SCFI_Index and Europe_Base_Price are properly excluded")

print("\n3. EXPANDED TESTING")
print("   - Test multiple forecast horizons: 2-week, 3-week, 1-month")
print("   - Create performance degradation curve")
print("   - This will show if model truly captures patterns vs. just stickiness")

print("\n4. ECONOMIC VALIDATION")
print("   - Compare RMSE to typical freight rate volatility")
print("   - Consult domain experts on expected forecast accuracy")
print("   - Benchmark against industry forecasts if available")

print(f"\n" + "="*80)